# Notebook 03: Modelo Preditivo de Avaliações Negativas

**Projeto:** Predição de Avaliações Negativas no Ecossistema Olist<br>
**Autor:** Rafael de Menezes Ehlers<br>
**Fase:** 2 (Implementação)<br>
**Curso:** Curso Superior de Tecnologia em Banco de Dados

## Objetivo

Treinar e avaliar dois modelos de classificação supervisionada para prever a variável-alvo `avaliacao_negativa` (1 se `review_score <= 3`, 0 caso contrário):

- **Regressão Logística** como baseline interpretável.
- **Random Forest** como modelo principal, mais robusto a interações não-lineares.

Ao final, a análise de **importância de features** do Random Forest responde diretamente à pergunta de pesquisa da Fase 1:

> Quais fatores operacionais, logísticos e financeiros possuem maior peso estatístico na determinação de uma avaliação negativa por parte do cliente no e-commerce brasileiro?

## Saída

- Métricas comparativas (acurácia, precisão, recall, F1, ROC-AUC) salvas no relatório.
- Modelo Random Forest serializado em `modelo_random_forest.pkl` para reutilização futura.


## 1. Setup do ambiente

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Caminhos
BASE_PATH = '/content/drive/MyDrive/PUCRS/projetobi-olist'
DW_PATH = os.path.join(BASE_PATH, 'olist_dw.sqlite')
MODEL_PATH = os.path.join(BASE_PATH, 'modelo_random_forest.pkl')
engine = create_engine(f'sqlite:///{DW_PATH}')

print(f'Data Warehouse: {DW_PATH}')
print(f'Modelo a ser salvo em: {MODEL_PATH}')

Data Warehouse: /content/drive/MyDrive/PUCRS/projetobi-olist/olist_dw.sqlite
Modelo a ser salvo em: /content/drive/MyDrive/PUCRS/projetobi-olist/modelo_random_forest.pkl


## 2. Carregamento dos dados

Consulta a fato `fato_pedidos` junto com a `dim_produto` para incluir a categoria do produto, que entra como uma das variáveis preditoras.

In [3]:
df = pd.read_sql('''
SELECT
    f.delta_entrega_dias,
    f.atraso,
    f.tempo_total_entrega_dias,
    f.tempo_consolidacao_dias,
    f.valor_total_pedido,
    f.valor_total_frete,
    f.razao_frete_preco,
    f.num_itens,
    f.num_sellers_unicos,
    f.multi_vendedor,
    f.faixa_ticket,
    f.avaliacao_negativa,
    p.categoria,
    c.estado_cliente
FROM fato_pedidos f
LEFT JOIN dim_produto p ON f.produto_key = p.product_id
LEFT JOIN dim_cliente c ON f.cliente_key = c.customer_id
''', engine)

print(f'Linhas carregadas: {len(df):,}')
print(f'Colunas: {list(df.columns)}')
print(f'\nValores nulos por coluna:')
print(df.isna().sum()[df.isna().sum() > 0])

Linhas carregadas: 95,824
Colunas: ['delta_entrega_dias', 'atraso', 'tempo_total_entrega_dias', 'tempo_consolidacao_dias', 'valor_total_pedido', 'valor_total_frete', 'razao_frete_preco', 'num_itens', 'num_sellers_unicos', 'multi_vendedor', 'faixa_ticket', 'avaliacao_negativa', 'categoria', 'estado_cliente']

Valores nulos por coluna:
tempo_consolidacao_dias      15
categoria                  1350
dtype: int64


In [4]:
# Remover linhas com nulos nas features criticas para o modelo
antes = len(df)
df = df.dropna(subset=[
    'delta_entrega_dias', 'tempo_total_entrega_dias', 'tempo_consolidacao_dias',
    'razao_frete_preco', 'categoria', 'estado_cliente'
])
print(f'Linhas antes do dropna: {antes:,}')
print(f'Linhas após dropna:    {len(df):,}')
print(f'Perda:                 {100 * (1 - len(df) / antes):.2f}%')

Linhas antes do dropna: 95,824
Linhas após dropna:    94,460
Perda:                 1.42%


## 3. Engenharia de features para ML

Três grupos de variáveis preditoras:

- **Numéricas** (10): valores contínuos ou binários já em formato 0/1.
- **Categóricas** (3): `faixa_ticket`, `estado_cliente`, `categoria` → convertidas via One-Hot Encoding.

Para reduzir a dimensionalidade da feature `categoria` (que tem cerca de 70 valores distintos), mantemos apenas as **15 categorias mais frequentes** e agrupamos as demais em `outras`.

In [5]:
# Reduzir cardinalidade da categoria
top_categorias = df['categoria'].value_counts().head(15).index
df['categoria_reduzida'] = df['categoria'].where(df['categoria'].isin(top_categorias), 'outras')

print(f'Categorias originais: {df["categoria"].nunique()}')
print(f'Categorias após redução: {df["categoria_reduzida"].nunique()}')
print(f'\nDistribuição após redução:')
print(df['categoria_reduzida'].value_counts())

Categorias originais: 73
Categorias após redução: 16

Distribuição após redução:
categoria_reduzida
outras                    18908
cama_mesa_banho            9071
beleza_saude               8562
esporte_lazer              7444
informatica_acessorios     6469
moveis_decoracao           6163
utilidades_domesticas      5655
relogios_presentes         5430
telefonia                  4051
automotivo                 3774
brinquedos                 3748
cool_stuff                 3499
ferramentas_jardim         3394
perfumaria                 3063
bebes                      2741
eletronicos                2488
Name: count, dtype: int64


In [6]:
# Lista explicita de features
features_numericas = [
    'delta_entrega_dias', 'tempo_total_entrega_dias', 'tempo_consolidacao_dias',
    'valor_total_pedido', 'valor_total_frete', 'razao_frete_preco',
    'num_itens', 'num_sellers_unicos',
    'atraso', 'multi_vendedor'
]

features_categoricas = ['faixa_ticket', 'estado_cliente', 'categoria_reduzida']

# One-Hot Encoding (drop_first=True para evitar dummy variable trap na LogReg)
df_encoded = pd.get_dummies(
    df[features_numericas + features_categoricas + ['avaliacao_negativa']],
    columns=features_categoricas,
    drop_first=True
)

print(f'Shape após encoding: {df_encoded.shape}')
print(f'Total de features após encoding: {df_encoded.shape[1] - 1}')

Shape após encoding: (94460, 54)
Total de features após encoding: 53


In [7]:
# Separar X (features) e y (target)
X = df_encoded.drop(columns=['avaliacao_negativa'])
y = df_encoded['avaliacao_negativa']

# Divisao treino/teste 80/20 estratificada (preserva proporcao do alvo)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Conjunto de treino: {X_treino.shape[0]:,} pedidos')
print(f'Conjunto de teste:  {X_teste.shape[0]:,} pedidos')
print(f'\nProporção da classe positiva (avaliação negativa):')
print(f'  Treino: {y_treino.mean()*100:.2f}%')
print(f'  Teste:  {y_teste.mean()*100:.2f}%')

Conjunto de treino: 75,568 pedidos
Conjunto de teste:  18,892 pedidos

Proporção da classe positiva (avaliação negativa):
  Treino: 21.04%
  Teste:  21.04%


## 4. Modelo baseline: Regressão Logística

Pipeline com `StandardScaler` (necessário para que o solver lbfgs convirja bem) seguido de `LogisticRegression` com `class_weight='balanced'` para compensar o leve desbalanceamento das classes.

In [8]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

pipe_lr.fit(X_treino, y_treino)
y_pred_lr = pipe_lr.predict(X_teste)
y_proba_lr = pipe_lr.predict_proba(X_teste)[:, 1]

print('Regressão Logística treinada.')

Regressão Logística treinada.


In [9]:
print('=== Regressão Logística ===')
print(f'Acurácia:  {accuracy_score(y_teste, y_pred_lr):.4f}')
print(f'Precisão:  {precision_score(y_teste, y_pred_lr):.4f}')
print(f'Recall:    {recall_score(y_teste, y_pred_lr):.4f}')
print(f'F1-score:  {f1_score(y_teste, y_pred_lr):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_teste, y_proba_lr):.4f}')
print(f'\nClassification Report:\n{classification_report(y_teste, y_pred_lr, target_names=["Positiva", "Negativa"])}')

=== Regressão Logística ===
Acurácia:  0.7550
Precisão:  0.4257
Recall:    0.4704
F1-score:  0.4469
ROC-AUC:   0.6983

Classification Report:
              precision    recall  f1-score   support

    Positiva       0.85      0.83      0.84     14917
    Negativa       0.43      0.47      0.45      3975

    accuracy                           0.76     18892
   macro avg       0.64      0.65      0.64     18892
weighted avg       0.76      0.76      0.76     18892



## 5. Modelo principal: Random Forest

Conjunto de 200 árvores de decisão com profundidade máxima 15 e `class_weight='balanced'`. Random Forest tipicamente supera regressão logística em problemas com interações não-lineares e variáveis categóricas, ambos presentes neste dataset.

In [10]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_treino, y_treino)
y_pred_rf = rf.predict(X_teste)
y_proba_rf = rf.predict_proba(X_teste)[:, 1]

print('Random Forest treinado.')

Random Forest treinado.


In [11]:
print('=== Random Forest ===')
print(f'Acurácia:  {accuracy_score(y_teste, y_pred_rf):.4f}')
print(f'Precisão:  {precision_score(y_teste, y_pred_rf):.4f}')
print(f'Recall:    {recall_score(y_teste, y_pred_rf):.4f}')
print(f'F1-score:  {f1_score(y_teste, y_pred_rf):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_teste, y_proba_rf):.4f}')
print(f'\nClassification Report:\n{classification_report(y_teste, y_pred_rf, target_names=["Positiva", "Negativa"])}')

=== Random Forest ===
Acurácia:  0.7887
Precisão:  0.4975
Recall:    0.3975
F1-score:  0.4419
ROC-AUC:   0.7024

Classification Report:
              precision    recall  f1-score   support

    Positiva       0.85      0.89      0.87     14917
    Negativa       0.50      0.40      0.44      3975

    accuracy                           0.79     18892
   macro avg       0.67      0.65      0.66     18892
weighted avg       0.77      0.79      0.78     18892



## 6. Comparação dos modelos

In [12]:
comparativo = pd.DataFrame({
    'Metrica': ['Acurácia', 'Precisão', 'Recall', 'F1-score', 'ROC-AUC'],
    'Regressão Logística': [
        accuracy_score(y_teste, y_pred_lr),
        precision_score(y_teste, y_pred_lr),
        recall_score(y_teste, y_pred_lr),
        f1_score(y_teste, y_pred_lr),
        roc_auc_score(y_teste, y_proba_lr)
    ],
    'Random Forest': [
        accuracy_score(y_teste, y_pred_rf),
        precision_score(y_teste, y_pred_rf),
        recall_score(y_teste, y_pred_rf),
        f1_score(y_teste, y_pred_rf),
        roc_auc_score(y_teste, y_proba_rf)
    ]
})
comparativo['Diferença (RF - LR)'] = comparativo['Random Forest'] - comparativo['Regressão Logística']
print(comparativo.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

 Metrica  Regressão Logística  Random Forest  Diferença (RF - LR)
Acurácia               0.7550         0.7887               0.0337
Precisão               0.4257         0.4975               0.0718
  Recall               0.4704         0.3975              -0.0730
F1-score               0.4469         0.4419              -0.0050
 ROC-AUC               0.6983         0.7024               0.0041


In [13]:
# Curvas ROC sobrepostas
fpr_lr, tpr_lr, _ = roc_curve(y_teste, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_teste, y_proba_rf)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=fpr_lr, y=tpr_lr, mode='lines',
                              name=f'Regressao Logistica (AUC={roc_auc_score(y_teste, y_proba_lr):.3f})',
                              line=dict(color='#3b82f6', width=2)))
fig_roc.add_trace(go.Scatter(x=fpr_rf, y=tpr_rf, mode='lines',
                              name=f'Random Forest (AUC={roc_auc_score(y_teste, y_proba_rf):.3f})',
                              line=dict(color='#dc2626', width=2)))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Aleatorio (AUC=0.5)',
                              line=dict(color='gray', dash='dash')))
fig_roc.update_layout(
    title='Curvas ROC: Regressão Logística vs Random Forest',
    xaxis_title='Taxa de Falsos Positivos (1 - Especificidade)',
    yaxis_title='Taxa de Verdadeiros Positivos (Sensibilidade)',
    height=500, width=700
)
fig_roc

In [14]:
# Matrizes de confusao lado a lado
cm_lr = confusion_matrix(y_teste, y_pred_lr)
cm_rf = confusion_matrix(y_teste, y_pred_rf)

fig_cm = make_subplots(rows=1, cols=2,
                       subplot_titles=('Regressão Logística', 'Random Forest'))

for col, (cm, model_name) in enumerate([(cm_lr, 'LR'), (cm_rf, 'RF')], start=1):
    fig_cm.add_trace(
        go.Heatmap(
            z=cm,
            x=['Predito: Positiva', 'Predito: Negativa'],
            y=['Real: Positiva', 'Real: Negativa'],
            text=cm, texttemplate='%{text:,}', textfont={'size': 14},
            colorscale='Blues', showscale=(col == 2)
        ),
        row=1, col=col
    )

fig_cm.update_layout(title='Matrizes de confusão', height=400, width=900)
fig_cm

## 7. Importância das features (resposta à pergunta de pesquisa)

A análise de feature importance do Random Forest indica **quais variáveis tiveram maior peso estatístico** na predição de avaliações negativas. Esse é o output central que conecta o modelo às três hipóteses da Fase 1.

In [15]:
# Extrair importancia das features
importancias = pd.DataFrame({
    'feature': X.columns,
    'importancia': rf.feature_importances_
}).sort_values('importancia', ascending=False)

print('Top 20 features mais importantes:')
print(importancias.head(20).to_string(index=False))

Top 20 features mais importantes:
                           feature  importancia
                delta_entrega_dias     0.236745
          tempo_total_entrega_dias     0.169785
                            atraso     0.122173
                 valor_total_frete     0.072415
                 razao_frete_preco     0.063261
           tempo_consolidacao_dias     0.061525
                valor_total_pedido     0.059757
                         num_itens     0.057696
                    multi_vendedor     0.021017
                num_sellers_unicos     0.017466
                 estado_cliente_SP     0.007127
                 estado_cliente_RJ     0.006270
categoria_reduzida_cama_mesa_banho     0.006161
                faixa_ticket_medio     0.006105
         categoria_reduzida_outras     0.005086
  categoria_reduzida_esporte_lazer     0.004392
                 estado_cliente_MG     0.004221
   categoria_reduzida_beleza_saude     0.004203
                faixa_ticket_baixo     0.004071
      

In [17]:
# Grafico das top 15 features
top15 = importancias.head(15).sort_values('importancia')

fig_imp = px.bar(
    top15, x='importancia', y='feature', orientation='h',
    title='Top 15 features mais importantes (Random Forest)',
    labels={'importancia': 'Importância relativa', 'feature': 'Feature'},
    color='importancia', color_continuous_scale='Blues'
)
fig_imp.update_layout(height=550, showlegend=False, coloraxis_showscale=False)
fig_imp

### Interpretação à luz das 3 hipóteses

Mapeamento das features mais relevantes para as hipóteses originais da Fase 1:

| Feature | Hipótese | Variável relacionada |
|---|---|---|
| `delta_entrega_dias`, `atraso`, `tempo_total_entrega_dias` | **H1** | Atraso na entrega |
| `razao_frete_preco`, `valor_total_frete`, `faixa_ticket_*` | **H2** | Frete em baixo ticket |
| `multi_vendedor`, `num_sellers_unicos`, `tempo_consolidacao_dias` | **H3** | Pedidos multi-vendedor |

A posição relativa dessas features no ranking de importância indica quais dos três mecanismos investigados têm maior poder explicativo na insatisfação do cliente Olist.

## 8. Inferência de exemplo

Aplicação prática do modelo: dado um pedido sintético com características específicas, qual a probabilidade de receber uma avaliação negativa? Comparamos dois cenários, um de baixo risco e outro de alto risco.

In [18]:
# Funcao auxiliar para construir um pedido sintetico no formato esperado pelo modelo
def montar_pedido(delta_entrega, tempo_entrega, tempo_consolidacao,
                  valor_pedido, valor_frete, num_itens, num_sellers,
                  faixa_ticket, estado, categoria):
    base = {col: 0 for col in X.columns}  # zera todas as colunas

    # Numericas
    base['delta_entrega_dias'] = delta_entrega
    base['tempo_total_entrega_dias'] = tempo_entrega
    base['tempo_consolidacao_dias'] = tempo_consolidacao
    base['valor_total_pedido'] = valor_pedido
    base['valor_total_frete'] = valor_frete
    base['razao_frete_preco'] = valor_frete / valor_pedido if valor_pedido > 0 else 0
    base['num_itens'] = num_itens
    base['num_sellers_unicos'] = num_sellers
    base['atraso'] = 1 if delta_entrega > 0 else 0
    base['multi_vendedor'] = 1 if num_sellers > 1 else 0

    # Categoricas (one-hot)
    col_faixa = f'faixa_ticket_{faixa_ticket}'
    if col_faixa in base: base[col_faixa] = 1
    col_estado = f'estado_cliente_{estado}'
    if col_estado in base: base[col_estado] = 1
    col_cat = f'categoria_reduzida_{categoria}'
    if col_cat in base: base[col_cat] = 1

    return pd.DataFrame([base])

In [19]:
# Cenario A: pedido de BAIXO risco
pedido_baixo_risco = montar_pedido(
    delta_entrega=-5,        # entregue 5 dias antes do prazo
    tempo_entrega=8,
    tempo_consolidacao=2,
    valor_pedido=350,        # ticket alto
    valor_frete=15,          # frete leve
    num_itens=1,
    num_sellers=1,           # single-vendedor
    faixa_ticket='alto',
    estado='SP',
    categoria='cama_mesa_banho'
)

# Cenario B: pedido de ALTO risco
pedido_alto_risco = montar_pedido(
    delta_entrega=12,        # atrasou 12 dias
    tempo_entrega=25,
    tempo_consolidacao=10,
    valor_pedido=30,         # ticket baixo
    valor_frete=25,          # frete pesado (razao 0.83)
    num_itens=3,
    num_sellers=2,           # multi-vendedor
    faixa_ticket='baixo',
    estado='AM',             # estado distante (norte)
    categoria='outras'
)

prob_baixo = rf.predict_proba(pedido_baixo_risco)[0, 1]
prob_alto = rf.predict_proba(pedido_alto_risco)[0, 1]

print(f'Probabilidade de avaliação negativa:')
print(f'  Cenário A (baixo risco): {prob_baixo*100:.1f}%')
print(f'  Cenário B (alto risco):  {prob_alto*100:.1f}%')
print(f'\nDiferença: {(prob_alto - prob_baixo)*100:.1f} pontos percentuais')

Probabilidade de avaliação negativa:
  Cenário A (baixo risco): 37.3%
  Cenário B (alto risco):  74.7%

Diferença: 37.5 pontos percentuais


## 9. Persistência do modelo

Serializa o modelo Random Forest treinado e a lista de colunas esperadas para uso futuro (por exemplo, integração com um sistema de monitoramento em tempo real).

In [20]:
artefato = {
    'modelo': rf,
    'colunas': list(X.columns),
    'metricas': {
        'acuracia': accuracy_score(y_teste, y_pred_rf),
        'precisao': precision_score(y_teste, y_pred_rf),
        'recall': recall_score(y_teste, y_pred_rf),
        'f1': f1_score(y_teste, y_pred_rf),
        'roc_auc': roc_auc_score(y_teste, y_proba_rf)
    }
}

joblib.dump(artefato, MODEL_PATH)
print(f'Modelo persistido em: {MODEL_PATH}')
print(f'Tamanho do arquivo: {os.path.getsize(MODEL_PATH) / (1024**2):.2f} MB')

Modelo persistido em: /content/drive/MyDrive/PUCRS/projetobi-olist/modelo_random_forest.pkl
Tamanho do arquivo: 53.14 MB
